## Sprites

In [4]:
import arcade
import random
import math

SCREEN_WIDTH = 800
SCREEN_HEIGHT = 600
SCREEN_TITLE = "Sprites"

SAFE_W = 120 # width of each safe zone
ENEMY_MIN_S, ENEMY_MAX_S = 120, 220 # enemy speed range
SAFE_ZONE_MAX_TIME = 15.0 # max seconds allowed in safe zone

class SpritePlayground(arcade.Window):
    def __init__(self):
        super().__init__(SCREEN_WIDTH, SCREEN_HEIGHT, SCREEN_TITLE)
        arcade.set_background_color(arcade.color.PALE_SPRING_BUD)

        self.player_list = arcade.SpriteList()
        self.enemy_list = arcade.SpriteList()

        self.player = arcade.Sprite(":resources:/images/animated_characters/zombie/zombie_fall.png", scale=0.6)
        self.player.center_x = 80
        self.player.center_y = SCREEN_HEIGHT // 2
        self.player.angle = 0
        self.player_list.append(self.player)

        # movement and speed
        self.speed = 220.0
        self.vx = 0.0
        self.vy = 0.0

        # key states
        self.left_down = False
        self.right_down = False
        self.up_down = False
        self.down_down = False

        # others
        self.score = 0
        self.last_zone = "left"
        self.game_over = False

        self._spawn_enemy()

    def _spawn_enemy(self):
        enemy = arcade.Sprite(":resources:/images/space_shooter/meteorGrey_small2.png", scale=0.8)
        enemy.center_x = random.randint(SAFE_W + 40, SCREEN_WIDTH - SAFE_W - 40)
        enemy.center_y = random.randint(40, SCREEN_HEIGHT - 40)
        angle = random.random() * 2 * math.pi
        speed = random.uniform(ENEMY_MIN_S, ENEMY_MAX_S)
        enemy.change_x = math.cos(angle) * speed
        enemy.change_y = math.sin(angle) * speed
        self.enemy_list.append(enemy)

    def on_draw(self):
        self.clear()

        # draw safe zones
        arcade.draw_rect_filled(
            arcade.rect.XYWH(SAFE_W / 2, SCREEN_HEIGHT / 2, SAFE_W, SCREEN_HEIGHT),
            arcade.color.CELADON
        )
        arcade.draw_rect_filled(
            arcade.rect.XYWH(SCREEN_WIDTH - SAFE_W / 2, SCREEN_HEIGHT / 2, SAFE_W, SCREEN_HEIGHT),
            arcade.color.LIGHT_STEEL_BLUE
        )
        
        self.player_list.draw()
        self.enemy_list.draw()

    def on_key_press(self, symbol, modifiers):
        if symbol in (arcade.key.LEFT, arcade.key.A):
            self.left_down = True
        if symbol in (arcade.key.RIGHT, arcade.key.D):
            self.right_down = True
        if symbol in (arcade.key.UP, arcade.key.W):
            self.up_down = True
        if symbol in (arcade.key.DOWN, arcade.key.S):
            self.down_down = True

    def on_key_release(self, symbol, modifiers):
        if symbol in (arcade.key.LEFT, arcade.key.A):
            self.left_down = False
        if symbol in (arcade.key.RIGHT, arcade.key.D):
            self.right_down = False
        if symbol in (arcade.key.UP, arcade.key.W):
            self.up_down = False
        if symbol in (arcade.key.DOWN, arcade.key.S):
            self.down_down = False

    def on_update(self, dt):
        # movement input
        dx = (1 if self.right_down else 0) - (1 if self.left_down else 0)
        dy = (1 if self.up_down else 0) - (1 if self.down_down else 0)

        if dx and dy:
            inv = 1 / 2**0.5
            dx *= inv
            dy *= inv

        self.vx = dx * self.speed
        self.vy = dy * self.speed

        self.player.center_x += self.vx * dt
        self.player.center_y += self.vy * dt

        if self.vx or self.vy:
            self.player.angle = -math.degrees(math.atan2(self.vy, self.vx))

        self.player.center_x = max(self.player.width / 2, min(SCREEN_WIDTH - self.player.width / 2, self.player.center_x))
        self.player.center_y = max(self.player.height / 2, min(SCREEN_HEIGHT - self.player.height / 2, self.player.center_y))

        left_wall = SAFE_W
        right_wall = SCREEN_WIDTH - SAFE_W

        # enemy movement
        for e in self.enemy_list:
            e.center_x += e.change_x * dt
            e.center_y += e.change_y * dt
            if e.left < left_wall:
                e.left = left_wall
                e.change_x *= -1
            elif e.right > right_wall:
                e.right = right_wall
                e.change_x *= -1
            if e.bottom < 0:
                e.bottom = 0
                e.change_y *= -1
            elif e.top > SCREEN_HEIGHT:
                e.top = SCREEN_HEIGHT
                e.change_y *= -1

        if arcade.check_for_collision_with_list(self.player, self.enemy_list):
            self.game_over = True
            return
try:
    window = SpritePlayground()
    arcade.run()
finally:
    arcade.close_window()

In [5]:
'''With timer'''

import arcade
import math
import random

SCREEN_WIDTH, SCREEN_HEIGHT = 800, 600
TITLE = "Avoid the Enemies – Laps & Safe Zones"

SAFE_W = 120  # width of each safe zone (left & right)
ENEMY_MIN_S, ENEMY_MAX_S = 120, 220  # enemy speed range (px/s)
SAFE_ZONE_MAX_TIME = 15.0  # Max seconds allowed camping in safe zone

class KeyboardOnlyDemo(arcade.Window):
    def __init__(self):
        super().__init__(SCREEN_WIDTH, SCREEN_HEIGHT, TITLE)
        arcade.set_background_color(arcade.color.PALE_SPRING_BUD)

        # ---- Player ----
        self.player_list = arcade.SpriteList()
        self.player = arcade.Sprite(":resources:images/animated_characters/female_person/femalePerson_idle.png", scale=1.0)
        self.player.center_x, self.player.center_y = 80, SCREEN_HEIGHT // 2
        self.player_list.append(self.player)

        # movement & speed
        self.speed = 220.0
        self.vx = 0.0
        self.vy = 0.0

        # key states
        self.left_down = False
        self.right_down = False
        self.up_down = False
        self.down_down = False
        self.sprint_down = False

        # ---- Enemies, score, state ----
        self.enemy_list = arcade.SpriteList()
        self.score = 0
        self.last_zone = "left"
        self.game_over = False

        # ---- Safe zone timer ----
        self.safe_zone_timer = 0.0
        self.in_safe_zone = False
        self.current_zone = None  # "left", "right" or None

        self._spawn_enemy()

    def _spawn_enemy(self):
        enemy = arcade.Sprite(":resources:images/space_shooter/meteorGrey_small1.png", scale=0.8)
        enemy.center_x = random.randint(SAFE_W + 40, SCREEN_WIDTH - SAFE_W - 40)
        enemy.center_y = random.randint(40, SCREEN_HEIGHT - 40)
        angle = random.random() * 2 * math.pi
        speed = random.uniform(ENEMY_MIN_S, ENEMY_MAX_S)
        enemy.change_x = math.cos(angle) * speed
        enemy.change_y = math.sin(angle) * speed
        self.enemy_list.append(enemy)

    def _reset(self):
        self.player.center_x, self.player.center_y = 80, SCREEN_HEIGHT // 2
        self.vx = self.vy = 0.0
        self.enemy_list = arcade.SpriteList()
        self.score = 0
        self.last_zone = "left"
        self.game_over = False
        self.left_down = self.right_down = self.up_down = self.down_down = self.sprint_down = False
        self.safe_zone_timer = 0.0
        self.in_safe_zone = False
        self.current_zone = None
        self._spawn_enemy()

    def on_draw(self):
        self.clear()

        # Safe zones
        arcade.draw_rect_filled(
            arcade.rect.XYWH(SAFE_W / 2, SCREEN_HEIGHT / 2, SAFE_W, SCREEN_HEIGHT),
            arcade.color.CELADON
        )
        arcade.draw_rect_filled(
            arcade.rect.XYWH(SCREEN_WIDTH - SAFE_W / 2, SCREEN_HEIGHT / 2, SAFE_W, SCREEN_HEIGHT),
            arcade.color.LIGHT_STEEL_BLUE
        )

        self.enemy_list.draw()
        self.player_list.draw()

        # HUD
        arcade.draw_text(f"Speed: {int(self.speed)} px/s", 16, SCREEN_HEIGHT - 32, arcade.color.BLACK, 16)
        arcade.draw_text(
            "WASD/Arrows move • Shift/Space sprint • +/- speed • R restart",
            16, 16, arcade.color.BLACK, 16
        )
        arcade.draw_text(f"Score: {self.score}   Enemies: {len(self.enemy_list)}",
                         16, SCREEN_HEIGHT - 58, arcade.color.BLACK, 18)

        # Safe zone timer display
        if self.in_safe_zone:
            remaining = max(0, SAFE_ZONE_MAX_TIME - self.safe_zone_timer)
            arcade.draw_text(f"Safe Zone Timer: {remaining:.1f}s",
                             SCREEN_WIDTH / 2, SCREEN_HEIGHT - 32,
                             arcade.color.DARK_RED if remaining < 5 else arcade.color.BLACK,
                             18, anchor_x="center")

        if self.game_over:
            arcade.draw_text("GAME OVER – Press R to restart",
                             SCREEN_WIDTH / 2, SCREEN_HEIGHT / 2 + 20,
                             arcade.color.DARK_RED, 28, anchor_x="center")
            arcade.draw_text(f"Final Score: {self.score}",
                             SCREEN_WIDTH / 2, SCREEN_HEIGHT / 2 - 16,
                             arcade.color.BLACK, 24, anchor_x="center")

    def on_key_press(self, symbol, modifiers):
        if symbol == arcade.key.R:
            self._reset()
            return
        if self.game_over:
            return

        if symbol in (arcade.key.LEFT, arcade.key.A):   self.left_down = True
        if symbol in (arcade.key.RIGHT, arcade.key.D):  self.right_down = True
        if symbol in (arcade.key.UP, arcade.key.W):     self.up_down = True
        if symbol in (arcade.key.DOWN, arcade.key.S):   self.down_down = True
        if symbol in (arcade.key.LSHIFT, arcade.key.RSHIFT, arcade.key.SPACE):
            self.sprint_down = True

        if symbol in (arcade.key.EQUAL, arcade.key.PLUS):
            self.speed = min(600, self.speed + 20)
        if symbol in (arcade.key.MINUS,):
            self.speed = max(60, self.speed - 20)

    def on_key_release(self, symbol: int, modifiers: int):
        if self.game_over:
            return

        if symbol in (arcade.key.LEFT, arcade.key.A):   self.left_down = False
        if symbol in (arcade.key.RIGHT, arcade.key.D):  self.right_down = False
        if symbol in (arcade.key.UP, arcade.key.W):     self.up_down = False
        if symbol in (arcade.key.DOWN, arcade.key.S):   self.down_down = False
        if symbol in (arcade.key.LSHIFT, arcade.key.RSHIFT, arcade.key.SPACE):
            self.sprint_down = False

    def on_update(self, dt: float):
        if self.game_over:
            return

        # movement input
        dx = (1 if self.right_down else 0) - (1 if self.left_down else 0)
        dy = (1 if self.up_down else 0) - (1 if self.down_down else 0)

        if dx and dy:
            inv = 1 / 2**0.5
            dx *= inv
            dy *= inv

        run_mult = 1.6 if self.sprint_down else 1.0
        self.vx = dx * self.speed * run_mult
        self.vy = dy * self.speed * run_mult

        self.player.center_x += self.vx * dt
        self.player.center_y += self.vy * dt

        if self.vx or self.vy:
            self.player.angle = -math.degrees(math.atan2(self.vy, self.vx))

        self.player.center_x = max(self.player.width / 2,
                                   min(SCREEN_WIDTH - self.player.width / 2, self.player.center_x))
        self.player.center_y = max(self.player.height / 2,
                                   min(SCREEN_HEIGHT - self.player.height / 2, self.player.center_y))

        left_wall = SAFE_W
        right_wall = SCREEN_WIDTH - SAFE_W

        # Enemy bounce
        for e in self.enemy_list:
            e.center_x += e.change_x * dt
            e.center_y += e.change_y * dt
            if e.left < left_wall:
                e.left = left_wall
                e.change_x *= -1
            elif e.right > right_wall:
                e.right = right_wall
                e.change_x *= -1
            if e.bottom < 0:
                e.bottom = 0
                e.change_y *= -1
            elif e.top > SCREEN_HEIGHT:
                e.top = SCREEN_HEIGHT
                e.change_y *= -1

        if arcade.check_for_collision_with_list(self.player, self.enemy_list):
            self.game_over = True
            return

        # Safe zone logic
        in_left = self.player.center_x <= SAFE_W
        in_right = self.player.center_x >= SCREEN_WIDTH - SAFE_W

        if in_left or in_right:
            if not self.in_safe_zone:
                self.in_safe_zone = True
                self.current_zone = "left" if in_left else "right"

            self.safe_zone_timer += dt

            if self.safe_zone_timer >= SAFE_ZONE_MAX_TIME:
                if self.current_zone == "left":
                    self.player.center_x = SAFE_W + self.player.width / 2 + 2
                else: 
                    self.player.center_x = SCREEN_WIDTH - SAFE_W - self.player.width / 2 - 2
                self.in_safe_zone = False
                self.current_zone = None

        else:
            self.in_safe_zone = False

        # Lap logic & timer reset on zone swap
        if in_right and self.last_zone == "left":
            self.score += 1
            self.last_zone = "right"
            self._spawn_enemy()
            self.safe_zone_timer = 0.0
        elif in_left and self.last_zone == "right":
            self.last_zone = "left"
            self.safe_zone_timer = 0.0
window = KeyboardOnlyDemo()
arcade.run()

C:\Users\miche\anaconda3\Lib\site-packages\arcade\exceptions.py:138: PerformanceWarning: draw_text is an extremely slow function for displaying text. Consider using Text objects instead.
  warnings.warn(message, warning_type)
